# Phase 3: Exploratory Data Analysis (EDA) — Track A (Lead)
**Project:** AI Stock Movement Prediction & Backtesting Platform  
**Objectives:**
1. **Price Dynamics & Return Profiles**: Analyze daily and cumulative returns across 2018–2026.
2. **Fat-Tail & Skewness**: Quantify departure from Gaussian normality (kurtosis, crash events).
3. **Volatility Regimes**: Identify high vs. low volatility regimes using rolling 20d/60d windows.
4. **Cross-Asset Correlation**: Analyze sector and inter-stock correlations for diversification insights.
5. **Volume Dynamics & Liquidity Spikes**: Identify abnormal trading activity (>2.5σ).

> **Anti-Leakage Rule:** All EDA indicators (MA, rolling std, returns) are strictly retrospective (backward-looking) and utilize only information available at time $t$.

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src.data.eda import run_eda_pipeline

%matplotlib inline
print("Starting EDA Pipeline...")

## 1. Run Pipeline & Compute Summary Statistics

In [ ]:
df_metrics, stats_df = run_eda_pipeline(
    data_path="../data/processed/combined_market_data.csv",
    report_dir="../reports"
)
stats_df

## 2. Visual Inspection: Cumulative Returns

In [ ]:
plt.figure(figsize=(14, 6))
for ticker, grp in df_metrics.groupby('ticker'):
    plt.plot(pd.to_datetime(grp['date']), grp['cumulative_return'] * 100, label=ticker, lw=2)
plt.title("Cumulative Returns Comparison (2018–2026)", fontsize=14, fontweight='bold')
plt.ylabel("Return (%)")
plt.xlabel("Date")
plt.legend()
plt.show()

## 3. Cross-Asset Return Correlation

In [ ]:
pivot_ret = df_metrics.pivot(index='date', columns='ticker', values='simple_return').dropna()
corr = pivot_ret.corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, cmap='coolwarm', vmin=0, vmax=1, fmt='.3f')
plt.title("Asset Return Correlation Heatmap", fontsize=13, fontweight='bold')
plt.show()

## 4. Key EDA Takeaways for Modeling (Track A & Track B):
1. **Heavy Tails**: Return distributions exhibit high excess kurtosis (> 5 for several tickers), indicating frequent extreme moves not captured by standard linear models without regularization.
2. **Banking Sector Correlation**: High correlation between `BBCA.JK`, `BBRI.JK`, and `BMRI.JK` (> 0.50), reflecting systemic banking sector momentum.
3. **Balanced Directional Target**: Up-day vs. down-day ratios are close to 50:50 (~50.5% - 51.5%), which means class imbalance is minimal, but distinguishing signal from noise will require careful calibration.